# LGBM Optuna Tuning

Tune `LGBMRegressor` with `Optuna` on top of a minimal set of row-wise features in the `log1p(target)` setup and optimize cross-validated `RMSLE`.

In [1]:
import sys

sys.path.append("../")

import json
from pathlib import Path

import numpy as np
import optuna
import pandas as pd

/Users/ayeustsihneyeu/ml_/santander/.santander/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error, root_mean_squared_log_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split

from src.features import add_rowwise_features
from src.loader import Loader
from src.modeling import build_lgbm_regressor

In [3]:
SEED = 42
TEST_SIZE = 0.33
CV = 5
N_TRIALS = 60

In [4]:
loader = Loader()
df = loader.load("../data/processed_data.csv")
df.shape

(4459, 4732)

In [5]:
X = df.drop(columns="target")
y = df["target"]
y_log = np.log1p(y)

(X.shape, y.shape)

((4459, 4731), (4459,))

Only a minimal feature set is added here: row sparsity, overall row magnitude, and the average/spread of non-zero values.

In [7]:
X = add_rowwise_features(X)
X.shape

(4459, 4739)

In [8]:
X_train, X_test, y_train_raw, y_test_raw, y_train_log, y_test_log = train_test_split(
    X,
    y,
    y_log,
    test_size=TEST_SIZE,
    random_state=SEED,
)

cv = KFold(n_splits=CV, shuffle=True, random_state=SEED)

The optimization target is RMSE in log-space. For the `log1p(target)` setup, this is equivalent to optimizing `RMSLE`. The search now runs on the feature space augmented with minimal row-wise aggregates.

In [ ]:
def objective(trial: optuna.Trial) -> float:
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 2000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 8, 128),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 120),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "subsample_freq": trial.suggest_int("subsample_freq", 1, 7),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 0.5),
    }

    model = build_lgbm_regressor(params)
    scores = -cross_val_score(
        estimator=model,
        X=X_train,
        y=y_train_log,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
    )
    return float(scores.mean())


In [10]:
study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

study.best_value

[I 2026-05-01 22:22:23,973] A new study created in memory with name: no-name-c215f305-b6fb-4e59-baac-8fe040c436f2
Best trial: 0. Best value: 1.46093:   2%|▏         | 1/60 [00:19<18:46, 19.10s/it]

[I 2026-05-01 22:22:43,097] Trial 0 finished with value: 1.4609285567114092 and parameters: {'n_estimators': 874, 'learning_rate': 0.08927180304353628, 'num_leaves': 96, 'max_depth': 8, 'min_child_samples': 23, 'subsample': 0.662397808134481, 'subsample_freq': 1, 'colsample_bytree': 0.9330880728874675, 'reg_alpha': 0.10129197956845731, 'reg_lambda': 0.34702669886504117, 'min_split_gain': 0.010292247147901223}. Best is trial 0 with value: 1.4609285567114092.


Best trial: 0. Best value: 1.46093:   3%|▎         | 2/60 [00:41<20:25, 21.13s/it]

[I 2026-05-01 22:23:05,645] Trial 1 finished with value: 1.4718618511797537 and parameters: {'n_estimators': 1946, 'learning_rate': 0.06798962421591129, 'num_leaves': 33, 'max_depth': 4, 'min_child_samples': 26, 'subsample': 0.721696897183815, 'subsample_freq': 4, 'colsample_bytree': 0.7159725093210578, 'reg_alpha': 0.0028585493941961923, 'reg_lambda': 0.11462107403425029, 'min_split_gain': 0.06974693032602092}. Best is trial 0 with value: 1.4609285567114092.


Best trial: 2. Best value: 1.38898:   5%|▌         | 3/60 [01:01<19:38, 20.67s/it]

...

Best trial: 45. Best value: 1.36229:  98%|█████████▊| 59/60 [15:00<00:22, 22.94s/it]

[I 2026-05-01 22:37:24,479] Trial 58 finished with value: 1.3638576028899148 and parameters: {'n_estimators': 322, 'learning_rate': 0.013183052610192113, 'num_leaves': 109, 'max_depth': 9, 'min_child_samples': 17, 'subsample': 0.74700917757097, 'subsample_freq': 7, 'colsample_bytree': 0.70285591292829, 'reg_alpha': 6.006863991724864, 'reg_lambda': 0.0005961285067222522, 'min_split_gain': 0.0543987938680692}. Best is trial 45 with value: 1.3622939737297948.


Best trial: 45. Best value: 1.36229: 100%|██████████| 60/60 [15:12<00:00, 15.20s/it]

[I 2026-05-01 22:37:36,052] Trial 59 finished with value: 1.363598862262888 and parameters: {'n_estimators': 316, 'learning_rate': 0.015361748501065095, 'num_leaves': 108, 'max_depth': 9, 'min_child_samples': 18, 'subsample': 0.7515821162376439, 'subsample_freq': 7, 'colsample_bytree': 0.7942400882591922, 'reg_alpha': 5.001248123908952, 'reg_lambda': 0.0017410421435952934, 'min_split_gain': 0.047365809514727346}. Best is trial 45 with value: 1.3622939737297948.


1.3622939737297948

In [10]:
best_params = study.best_params
best_params

{'n_estimators': 439,
 'learning_rate': 0.019291857433628438,
 'num_leaves': 32,
 'max_depth': 11,
 'min_child_samples': 5,
 'subsample': 0.9872554086372487,
 'subsample_freq': 5,
 'colsample_bytree': 0.7408418143236152,
 'reg_alpha': 0.025576812693216197,
 'reg_lambda': 0.718015922751547,
 'min_split_gain': 0.04545640466117268}

In [11]:
trials_df = study.trials_dataframe().sort_values(by="value").reset_index(drop=True)
trials_df.head(10)

,number,value,datetime_start,datetime_complete,duration,params_colsample_bytree,params_learning_rate,params_max_depth,params_min_child_samples,params_min_split_gain,params_n_estimators,params_num_leaves,params_reg_alpha,params_reg_lambda,params_subsample,params_subsample_freq,state
0,52,1.434652,2026-05-01 22:01:11.837551,2026-05-01 22:01:28.004862,0 days 00:00:16.167311,0.740842,0.019292,11,5,0.045456,439,32,0.025577,0.718016,0.987255,5,COMPLETE
1,50,1.439508,2026-05-01 22:00:39.041329,2026-05-01 22:00:55.825455,0 days 00:00:16.784126,0.736093,0.016028,11,5,0.051016,445,31,0.004475,0.004182,0.979955,5,COMPLETE
2,58,1.439525,2026-05-01 22:02:30.036199,2026-05-01 22:02:50.567166,0 days 00:00:20.530967,0.682730,0.015541,11,8,0.093833,472,52,0.158376,0.236434,0.984078,4,COMPLETE
3,22,1.439952,2026-05-01 21:50:26.320061,2026-05-01 21:50:48.488910,0 days 00:00:22.168849,0.814238,0.010517,12,5,0.087337,509,38,0.000309,0.000287,0.959095,7,COMPLETE
4,30,1.441487,2026-05-01 21:53:28.791595,2026-05-01 21:53:58.785712,0 days 00:00:29.994117,0.766100,0.019810,11,11,0.140595,943,38,5.168763,0.001705,0.881915,4,COMPLETE
5,51,1.441492,2026-05-01 22:00:55.829262,2026-05-01 22:01:11.834995,0 days 00:00:16.005733,0.740613,0.016401,11,6,0.055975,384,34,0.020127,0.221427,0.976980,5,COMPLETE
6,42,1.442080,2026-05-01 21:58:16.578122,2026-05-01 21:58:40.419281,0 days 00:00:23.841159,0.765290,0.016600,11,11,0.084394,659,48,0.000274,0.000107,0.940541,7,COMPLETE
7,27,1.442660,2026-05-01 21:52:07.704365,2026-05-01 21:52:42.617271,0 days 00:00:34.912906,0.768319,0.019303,11,7,0.127119,1168,39,0.000345,0.000964,0.947764,6,COMPLETE
8,54,1.443143,2026-05-01 22:01:40.636348,2026-05-01 22:01:53.138383,0 days 00:00:12.502035,0.703481,0.018144,11,5,0.045146,320,31,0.020966,0.861331,0.997600,4,COMPLETE
9,59,1.443435,2026-05-01 22:02:50.569655,2026-05-01 22:03:06.772309,0 days 00:00:16.202654,0.629808,0.013768,10,7,0.049934,443,33,0.063620,0.069317,0.987334,3,COMPLETE


In [12]:
best_model = build_lgbm_regressor(best_params)
best_model.fit(X_train, y_train_log)

y_pred_log = best_model.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_pred = np.clip(y_pred, 0, None)

In [13]:
metrics = pd.DataFrame(
    {
        "metric": ["cv_rmsle_mean", "holdout_rmsle", "holdout_rmse", "holdout_mae", "holdout_r2"],
        "value": [
            study.best_value,
            root_mean_squared_log_error(y_test_raw, y_pred),
            root_mean_squared_error(y_test_raw, y_pred),
            mean_absolute_error(y_test_raw, y_pred),
            r2_score(y_test_raw, y_pred),
        ],
    }
)

metrics.style.format({"value": "{:,.4f}"})

,metric,value
0,cv_rmsle_mean,1.4347
1,holdout_rmsle,1.4644
2,holdout_rmse,"7,357,560.5649"
3,holdout_mae,"4,128,461.5326"
4,holdout_r2,0.1518


In [14]:
ARTIFACTS_DIR = Path("../artifacts/optuna_lgbm")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

trials_df.to_csv(ARTIFACTS_DIR / "optuna_trials.csv", index=False)
metrics.to_csv(ARTIFACTS_DIR / "optuna_metrics.csv", index=False)

summary = {
    "target_transform": "log1p",
    "primary_metric": "rmsle",
    "optimizer": "optuna",
    "n_trials": N_TRIALS,
    "best_cv_rmsle": float(study.best_value),
    "best_params": best_params,
    "holdout_rmsle": float(root_mean_squared_log_error(y_test_raw, y_pred)),
    "holdout_rmse": float(root_mean_squared_error(y_test_raw, y_pred)),
    "holdout_mae": float(mean_absolute_error(y_test_raw, y_pred)),
    "holdout_r2": float(r2_score(y_test_raw, y_pred)),
}

with open(ARTIFACTS_DIR / "optuna_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

## How To Use

- Start with `N_TRIALS = 60` for a quick search.
- Increase the trial count only after confirming that the search is stable.
- Compare this notebook with the previous `LGBM` baseline by `cv_rmsle_mean` first, then by holdout `RMSLE`.